<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Appendix B: *WildFires*
- *Version Number: 4.0*
---
### Contents  
> 1. *Wildfire Count Data* 
> 2. *Wildfire Damage Data*
> 3. *Combine Datasets*
> 4. *Export File*
---
### Notes
---
### Inputs
> - **Calfire Incident** Data: <https://www.fire.ca.gov/incidents>
>   - raw dataset = '.../data/raw/mapdatall.csv'
---
### Outputs  
- `fire_data.csv` - cleaned dataset of fire related variables 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

### Third Party Dependencies  
---

In [2]:
# Core data handling
import pandas as pd
import datetime as dt
import numpy as np
import geopandas as gpd

# Plotting
import matplotlib.pyplot as plt
import geopandas as gpd

## 1. Total Fire Count

### Load `CALfires.csv` - Wildfire Incident Data
Data obtained from [CAL FIRE Incident Data](https://www.fire.ca.gov/incidents)\
Dates of fire damage information: 01/01/2018 - 01/23/2025

> Variables Used:
> - `Fire name`
> - `Date started` and `Date Out` - Start and extinquishing date and time of wildfire incident 
> - `Total_Cost_Estimated` - in dollar amount
> - `County`, `Latitude` and `Longitude` - Spatial factors

Workflow:
- filter dataset by wildfire type
- column formatting
- fill NA values
- Ensure spatial accuracy of coordinates
- Format date and time columns

In [3]:
# Load data
allfires = pd.read_csv("../data/raw/Calfires.csv",low_memory = False)
allfires

,Unnamed: 0,SourceOID,ContainmentDateTime,IncidentSize,EstimatedCostToDate,FinalAcres,FireBehaviorGeneral,FireBehaviorGeneral1,FireBehaviorGeneral2,FireBehaviorGeneral3,...,LocalIncidentIdentifier,POOCity,POOCounty,POOState,PredominantFuelGroup,PredominantFuelModel,PrimaryFuelModel,SecondaryFuelModel,UniqueFireIdentifier,EstimatedFinalCost
0,0,7747595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,066100,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2020-CALAC-066100,NaN
1,1,6384391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,009269,NaN,San Diego,US-CA,NaN,NaN,NaN,NaN,2019-CAMVU-009269,NaN
2,3,22499589,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,163915,NaN,Riverside,US-CA,NaN,NaN,NaN,NaN,2021-CARRU-163915,NaN
3,4,23869477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,396331,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2022-CALAC-396331,NaN
4,17,23247195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,008436,NaN,San Luis Obispo,US-CA,NaN,NaN,NaN,NaN,2020-CASLU-008436,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84534,364796,38438381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,242944,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-242944,NaN
84535,364802,38438397,7/13/2025 4:02:20 PM,1.7,NaN,NaN,NaN,NaN,NaN,NaN,...,020730,NaN,Sacramento,US-CA,NaN,NaN,NaN,NaN,2025-CAAEU-020730,NaN
84536,364803,38438398,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,...,000813,NaN,Glenn,US-CA,NaN,NaN,NaN,NaN,2025-CAMNF-000813,NaN
84537,364819,38438435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,243075,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-243075,NaN


In [4]:
# Filter for wildfire incidents only
wildfires = allfires[allfires['IncidentTypeCategory']=='WF']

# rename for convenience
wildfires = wildfires.rename(columns={
    'InitialLatitude': 'Fire_Latitude',
    'InitialLongitude': 'Fire_Longitude',
    'FireDiscoveryDateTime':'Date',
    'POOCounty':'County'
})

# Drop entries with crucial null variables, may be manually fixed soon
wildfires = wildfires.dropna(subset=['Fire_Latitude', 'Fire_Longitude','Date'])


# Filter Coordinates
## Keep only coordinates that fall inside California boundaries since
## some entries have incorrect latitude and longitude values. 
## To be manually corrected in the future but are filtered out for now.

wildfires = wildfires[
    (wildfires['Fire_Latitude'] >= 32.5) & 
    (wildfires['Fire_Latitude'] <= 42) &
    (wildfires['Fire_Longitude'] >= -124.5) &
    (wildfires['Fire_Longitude'] <= -114)
]

# Format date and time columns
wildfires['Date'] = pd.to_datetime(wildfires['Date']).dt.date
wildfires = wildfires.sort_values(by = 'Date')
                                  
# Fill Remaining NA values
## Entries with no cost present are assumed to have a damage cost 0 dollars.
wildfires['EstimatedFinalCost'] = wildfires['EstimatedFinalCost'].fillna(0)

drop =['Unnamed: 0', 'SourceOID', 'ContainmentDateTime','FinalAcres','FireBehaviorGeneral','FireBehaviorGeneral1', 'FireBehaviorGeneral2', 'FireBehaviorGeneral3',
 'FireCause', 'FireCauseGeneral', 'FireCauseSpecific','IncidentShortDescription', 'IncidentTypeKind',
  'IrwinID','LocalIncidentIdentifier', 'POOCity', 'POOState','PredominantFuelGroup', 'PredominantFuelModel', 
  'PrimaryFuelModel','SecondaryFuelModel', 'UniqueFireIdentifier','IncidentTypeCategory','FireOutDateTime','EstimatedFinalCost'
      ]

all_fires = wildfires.drop(columns=drop)

## Change Fire Name Field
all_fires['Fire Name'] = all_fires['IncidentName'].str.lower()
fire_data = all_fires.drop(columns='IncidentName')

In [5]:
#all_fires = all_fires.fillna(0)

### Only utilize fires with damage statistics

In [6]:
#fire_data = all_fires[all_fires['EstimatedFinalCost'].notna()]

In [7]:
fire_data = fire_data.rename(columns={'EstimatedCostToDate':'Estimated_Damage',
                                      'IncidentSize':'Acres',
                                     })

In [8]:
fire_data["Estimated_Damage_has_data"] = fire_data["Estimated_Damage"].notna().astype(int)
fire_data['Estimated_Damage'] = fire_data['Estimated_Damage'].fillna(0)

fire_data["Acres_has_data"] = fire_data["Acres"].notna().astype(int)
fire_data['Acres'] = fire_data['Acres'].fillna(0)

## Expand fire data over days burned

In [9]:
fire_data['Acres'].describe()

count     76791.000000
mean        126.613251
std        5935.219273
min           0.000000
25%           0.000000
50%           0.000000
75%           0.060000
max      963309.000000
Name: Acres, dtype: float64

In [10]:
fire_data['Meters']  = (fire_data['Acres'] * 4046.8564224) ** 0.5

In [11]:
fire_data['Meters'].describe()

count    76791.000000
mean        50.278305
std        714.047883
min          0.000000
25%          0.000000
50%          0.000000
75%         15.582406
max      62436.953909
Name: Meters, dtype: float64

In [12]:
fire_data['buffer'] = 23000.0

In [13]:
fire_data.loc[fire_data['Meters']>23000,'buffer'] = fire_data['Meters']

In [14]:
fire_data = fire_data.drop(columns=['County','Meters'])

In [15]:
fire_data

,Acres,Estimated_Damage,Date,Fire_Latitude,Fire_Longitude,Fire Name,Estimated_Damage_has_data,Acres_has_data,buffer
29874,0.00,0.0,2018-01-01,34.031731,-117.950401,lac-393275,0,0,23000.0
44288,0.10,0.0,2018-01-01,34.560180,-115.642300,tire,0,1,23000.0
9399,0.00,0.0,2018-01-01,34.682388,-118.085892,lac-393502,0,0,23000.0
34060,0.00,0.0,2018-01-01,33.930191,-118.343903,lac-393384,0,0,23000.0
5358,0.00,0.0,2018-01-01,33.985970,-118.174149,lac-393431,0,0,23000.0
...,...,...,...,...,...,...,...,...,...
84513,0.00,0.0,2025-07-13,34.671700,-118.127710,lac-242317,0,0,23000.0
84512,0.00,0.0,2025-07-13,34.363060,-118.504500,white,0,0,23000.0
84511,0.01,0.0,2025-07-13,37.345361,-120.607489,atwater bl / first st,0,1,23000.0
84523,0.07,0.0,2025-07-13,38.516749,-122.982029,river,0,1,23000.0


## 4. Export File

In [16]:
fire_data.to_csv("../data/processed/fire_data.csv",index=False)
print("All datasets saved successfully.")

All datasets saved successfully.


In [17]:
fire_data[fire_data['Estimated_Damage']>0]

,Acres,Estimated_Damage,Date,Fire_Latitude,Fire_Longitude,Fire Name,Estimated_Damage_has_data,Acres_has_data,buffer
38754,258.0,1200000.00,2018-03-27,33.989630,-119.720000,santa cruz,1,1,23000.0
42029,226.0,2200000.00,2018-04-24,40.705250,-123.544700,grape,1,1,23000.0
39780,1352.0,2900000.00,2018-06-04,34.519980,-118.281810,stone,1,1,23000.0
25235,175.0,1100000.00,2018-06-09,34.360360,-118.554300,south,1,1,23000.0
37576,12990.0,13919710.73,2018-06-11,37.557780,-119.127500,lions,1,1,23000.0
...,...,...,...,...,...,...,...,...,...
83398,105.0,2000000.00,2025-07-03,40.782991,-123.130903,helena,1,1,23000.0
83446,46.0,50000.00,2025-07-03,41.294333,-123.295667,jacket,1,1,23000.0
83493,5727.0,4470000.00,2025-07-04,41.315333,-123.410667,butler,1,1,23000.0
84066,403.0,2279155.00,2025-07-07,40.173203,-123.603917,bridge,1,1,23000.0
